# Platinum 4 Ejes — prueba de tiempos de refresco (COPI + SNP appendados)

**Pedido (Ezequiel Sueldo, vía Pablo Robledo, 2026-08-04):** una tabla de prueba en la capa
pre-agregada de BI para medir cómo se comporta el refresco de Power BI con la granularidad
pedida por Pricing, cruzada contra los 4 ejes del negocio:

- **Producto**: a nivel SKU (`Cod_SKU`) para la métrica simple; a nivel **par de EAN**
  (`EAN_Numerador`/`EAN_Denominador`) para el Índice de Precio — ver nota de alcance abajo.
- **Tiempo**: a nivel semana (`Fecha`, ya normalizada al lunes en Silver).
- **Cliente**: a nivel Subcanal para las 4 categorías (Supermercados, Mayoristas, E-Commerce,
  AASS + Petit) — **y además** a nivel Cliente (cadena) para los 3 Subcanales de Canal
  Moderno (Supermercados, Mayoristas, E-Commerce).
- **Geográfico**: "Región v2" (región entre Provincia y Localidad).

## Qué ya existía y qué no (relevado 2026-08-04)

Verificado contra el esquema real de `mdm.ICO_PDVs_Scanntech` y `mdm.ICO_SNP_Bocas`
(`INFORMATION_SCHEMA.COLUMNS`):

| Eje | Disponible hoy | Fuente |
|---|---|---|
| Producto (SKU) | Sí — `Cod_SKU`/`Descripcion_SKU` ya están en `Gold_Comparativo` | Ambas |
| Tiempo (semana) | Sí — `Fecha` ya viene semanal | Ambas |
| Cliente — Subcanal (4 categorías, incl. AASS+Petit) | Sí — `Subcanal Para` en `ICO_PDVs_Scanntech` | Solo **COPI** releva AASS+Petit |
| Cliente — Cliente (cadena) para Canal Moderno | Sí — `Cód Cli Marfrig`/`Descripción Cliente` en `ICO_SNP_Bocas` | Solo **SNP** identifica cadena |
| Geográfico — Región v2 (entre Provincia y Localidad) | **No existe** en ningún maestro (confirmado por `INFORMATION_SCHEMA.COLUMNS`, ninguna columna intermedia entre `Provincia` y `Localidad`/`Región` en ninguna de las dos fuentes) | — |

**Decisión (confirmada con el usuario, 2026-08-04):** hasta que Región v2 se defina a nivel
de maestro, esta tabla usa `Provincia` como placeholder en la columna `Region_v2`, marcado
explícitamente con el flag `Region_v2_Placeholder=1`. El día que el campo real exista, se
reemplaza en una sola columna sin tocar el resto del pipeline.

## Cómo se arma el "appendado"

Esta tabla NO es un join — es un `concat` de dos aperturas de grano distinto, mismo patrón que
ya usa `Gold_Comparativo` (COPI+SNP apilados) y Platinum COPI (multi-grano con centinela
`(Todos)`):

1. **Filas `Nivel_Cliente='Subcanal'`**: agregadas desde `Fuente='COPI'` — es la única fuente
   que releva las 4 categorías de Subcanal, incluyendo AASS+Petit (Canal Tradicional).
2. **Filas `Nivel_Cliente='Cliente'`**: agregadas desde `Fuente='SNP'` — es la única fuente que
   identifica la cadena (`Descripcion_Cliente`), y solo cubre Canal Moderno.

Ambas leen de `PRC.Gold_Comparativo` (no hace falta volver a Silver, la unificación de
producto/PDV ya está resuelta ahí).

**Fix aplicado (confirmado 2026-08-04 contra datos reales):** SNP trae `Subcanal` con un
prefijo numérico de orden (`"1. Supermercados"`, `"2. Mayoristas"`) que COPI no tiene. Se
normaliza en `leer_gold_comparativo()` antes de cualquier filtro, para que el matching de
Canal Moderno y la columna `Subcanal` de salida queden consistentes entre ambas fuentes.

**Hallazgo de negocio (no es un bug):** en los datos actuales, SNP no releva ninguna fila de
`E-Commerce` a nivel cadena — el panel SNP hoy solo cubre Supermercados y Mayoristas. La
apertura por Cliente para E-Commerce queda vacía hasta que ese relevamiento exista.

**Caveat de negocio a tener presente (no es un bug):** las filas de Subcanal (COPI) y las de
Cliente (SNP) provienen de paneles de relevamiento distintos. El precio de un Subcanal en esta
tabla NO va a coincidir exactamente con el agregado de sus Clientes componentes — son dos
mediciones independientes del mismo fenómeno, no una jerarquía padre-hijo sobre el mismo dato.

## Alcance de métricas — dos bloques apilados, discriminados por `Tipo_Metrica`

**Decisión (confirmada con el usuario, 2026-08-04):** el Índice de Precio ya está validado y
con luz verde de negocio (misma lógica que `platinum_price_index_copi.ipynb` /
`platinum_price_index_snp.ipynb` — mediana de cocientes por PDV entre pares de EAN), así que
se explosiona también al grano de 4 ejes, junto con la métrica simple original:

| `Tipo_Metrica` | Qué calcula | Eje Producto | Columnas de métrica pobladas |
|---|---|---|---|
| `Precio_Simple` | Promedio/mediana simple de `Precio_BASE` | SKU (`Cod_SKU`) | `Precio_Promedio`, `Precio_Mediana` |
| `Price_Index` | Mediana de cocientes por PDV entre un par de EAN (marca vs. marca propia) | **Par de EAN** (`Clave`/`EAN_Numerador`/`EAN_Denominador`) | `Price_Index`, `Price_Index_Nacional`, `PDG_KG_Num`, `PDG_KG_Den` |

**Por qué son dos bloques apilados y no una sola fila con todo:** el Índice de Precio no se
calcula "por SKU" — se calcula por PAR de SKUs (numerador/denominador), que es una unidad de
negocio distinta. Forzar las dos claves (SKU único vs. par de EAN) en la misma fila mezclaría
dos grafos de agregación distintos y volvería ambiguo el `GROUP BY` en Power BI. Se apilan con
el mismo patrón `(Todos)`/nulo que ya usa `Gold_Comparativo`: cada fila completa las columnas
de su propia métrica y deja en null las de la otra.

**Gap de gobierno heredado:** el Índice de Precio depende de `platinum_pairs_copi`,
`platinum_pairs_snp` y `platinum_precio_deflactado` — tablas de referencia pobladas por un
notebook no versionado en este repo (ver `HORUS_ROADMAP.md`). Si esas tablas no existen en el
ambiente donde se corre, el bloque `Price_Index` se omite con un aviso en vez de romper toda
la corrida (el bloque `Precio_Simple` no depende de ellas).

## Full rebuild

A diferencia de Platinum Price Index (incremental), esta tabla se reconstruye completa en cada
corrida (`if_exists='replace'`) — es una tabla chica de prueba, no vale la pena la complejidad
incremental todavía. Si el PoC de refresco confirma que el grano sirve, conviene pasarla a modo
incremental como el resto de Platinum.


In [1]:
# =============================================================================
# BOOTSTRAP: instalacion automatica de dependencias (self-contained, Linux-ready)
# =============================================================================
import importlib
import subprocess
import sys


def _bootstrap_dependencias(paquetes):
    faltantes = []
    for modulo, paquete_pip in paquetes.items():
        try:
            importlib.import_module(modulo)
        except ImportError:
            faltantes.append(paquete_pip)
    if faltantes:
        print(f"[INFO] Instalando dependencias faltantes: {', '.join(faltantes)}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltantes])


_bootstrap_dependencias({
    "pandas": "pandas",
    "numpy": "numpy",
    "sqlalchemy": "sqlalchemy",
    "pyodbc": "pyodbc",
})

# hasattr(...): bajo Jupyter/ipykernel, sys.stdout/stderr son un OutStream
# propio que NO tiene .reconfigure() (solo lo tiene un TextIOWrapper real,
# el caso de un script corrido por CPython en terminal/cron/systemd) -- sin
# este chequeo, la celda revienta con AttributeError apenas se corre como
# notebook en un kernel.
for _stream in (sys.stdout, sys.stderr):
    if hasattr(_stream, "reconfigure"):
        _stream.reconfigure(encoding="utf-8", errors="replace")

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import datetime


In [2]:
# =============================================================================
# CONFIGURACION (autocontenido: sin dependencias de otro archivo)
# =============================================================================

def _verificar_driver_odbc(nombre_driver="ODBC Driver 17 for SQL Server"):
    """
    El driver ODBC de SQL Server es un paquete de sistema operativo, no
    instalable por pip. Se valida su presencia antes de conectar para fallar
    con un mensaje claro en vez de un traceback criptico de pyodbc.
    """
    import pyodbc
    disponibles = pyodbc.drivers()
    if nombre_driver not in disponibles:
        raise RuntimeError(
            f"No se encontro el driver ODBC '{nombre_driver}' en este servidor.\n"
            f"Drivers disponibles: {disponibles or '(ninguno)'}\n"
            "Instalar en Debian/Ubuntu:\n"
            "  curl -sSL https://packages.microsoft.com/keys/microsoft.asc | sudo apt-key add -\n"
            "  curl -sSL https://packages.microsoft.com/config/ubuntu/22.04/prod.list | sudo tee /etc/apt/sources.list.d/mssql-release.list\n"
            "  sudo apt-get update && ACCEPT_EULA=Y sudo apt-get install -y msodbcsql17 unixodbc-dev"
        )


_verificar_driver_odbc()

# Conexion de trabajo: SANDBOX_BRONZE (Gold_Comparativo + tablas platinum).
conn_str_work = quote_plus(
    "DRIVER=ODBC Driver 17 for SQL Server;"
    "SERVER=10.1.82.27,1433;"
    "DATABASE=SANDBOX_BRONZE;"
    "UID=sa;"
    "PWD=M4rfr1g.2023*;"
    "TrustServerCertificate=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={conn_str_work}")

# Conexion al maestro de productos (MDS_PRODU): unica fuente del factor KG/U
# necesario para el Price Index (mismas credenciales que ya usan
# platinum_price_index_copi.ipynb / platinum_price_index_snp.ipynb).
conn_str_mds = quote_plus(
    "DRIVER=ODBC Driver 17 for SQL Server;"
    "SERVER=10.1.82.27\\mfgbi;"
    "DATABASE=MDS_PRODU;"
    "UID=pbi_report;"
    "PWD=Marfrig2024;"
    "TrustServerCertificate=yes;"
)
engine_mds = create_engine(f"mssql+pyodbc:///?odbc_connect={conn_str_mds}")

SCHEMA            = "PRC"
TABLA_GOLD        = "Gold_Comparativo"
TABLA_OUT         = "platinum_4ejes"
TABLA_PAIRS_COPI  = "platinum_pairs_copi"
TABLA_PAIRS_SNP   = "platinum_pairs_snp"
TABLA_DEFLACTADO  = "platinum_precio_deflactado"
TODOS             = "(Todos)"

# Subcanales que integran Canal Moderno (los unicos con apertura por Cliente).
# AASS + Petit (Canal Tradicional) queda excluido de la apertura por Cliente
# a proposito: SNP no lo releva a nivel cadena.
SUBCANALES_CANAL_MODERNO = ["Supermercados", "Mayoristas", "E-Commerce"]

# Claves que identifican un par de comparacion (constantes por fila; van en
# cada groupby de Price Index). Identico a PAIR_KEYS en los notebooks
# validados de Platinum.
PAIR_KEYS = [
    'Canal', 'Agrupador', 'Orden', 'Clave',
    'EAN_Numerador', 'Desc_Numerador', 'EAN_Denominador', 'Desc_Denominador',
    'Price_Index_Nacional',
]

# Orden y set final de columnas de salida. Las filas Precio_Simple no
# completan las columnas de par (EAN_Numerador...Price_Index); las filas
# Price_Index no completan Cod_SKU/Descripcion_SKU/Precio_Promedio/Precio_Mediana.
# Mismo patron que Gold_Comparativo: nulo cuando la fuente/metrica no aplica.
COLUMNAS_FINAL = [
    'Fecha', 'Fuente', 'Tipo_Metrica',
    'Cod_SKU', 'Descripcion_SKU',
    'Clave', 'EAN_Numerador', 'Desc_Numerador', 'EAN_Denominador', 'Desc_Denominador',
    'Region_v2', 'Region_v2_Placeholder', 'Provincia',
    'Subcanal', 'Nivel_Cliente', 'Cliente',
    'Precio_Promedio', 'Precio_Mediana',
    'Price_Index', 'Price_Index_Nacional', 'PDG_KG_Num', 'PDG_KG_Den',
    'Cantidad_PDV', 'Cantidad_Registros',
]


def _tabla_existe(engine, tabla):
    """
    Chequea existencia antes de leer platinum_pairs_copi/snp: son tablas de
    referencia con un gap de gobierno conocido (las puebla un notebook que no
    esta versionado en este repo, ver HORUS_ROADMAP.md) -- pueden no existir
    o estar vacias en un ambiente dado. Si falta la tabla, el Price Index se
    omite con un aviso en vez de romper toda la corrida.
    """
    with engine.connect() as conn:
        existe = conn.execute(text(
            f"SELECT 1 FROM sysobjects WHERE name='{tabla}' AND xtype='U'"
        )).fetchone()
    return existe is not None


def agregar_region_v2(df):
    """
    Region_v2 no existe todavia en ningun maestro (confirmado contra
    INFORMATION_SCHEMA.COLUMNS el 2026-08-04) -- se usa Provincia como
    placeholder, marcado con Region_v2_Placeholder=1 para que quede
    trazable en el modelo de Power BI. Compartido por el bloque de metricas
    simples y el de Price Index para no duplicar la logica del placeholder.
    """
    df = df.copy()
    df['Region_v2']             = df['Provincia']
    df['Region_v2_Placeholder'] = True
    return df


In [3]:
# =============================================================================
# FUNCIONES — PRICE INDEX (misma logica validada de
# platinum_price_index_copi.ipynb / platinum_price_index_snp.ipynb, reusada
# tal cual: mediana de cocientes por PDV entre pares de EAN. Aca solo se
# reagrupa al grano de 4 ejes en vez de al grano geografico/cadena original)
# =============================================================================

def leer_pairs(engine, tabla):
    """Lee los pares de comparacion (COPI o SNP) desde su tabla de referencia."""
    df = pd.read_sql(
        f"SELECT Clave, Canal, Agrupador, Orden, "
        f"EAN_Numerador, Desc_Numerador, EAN_Denominador, Desc_Denominador "
        f"FROM [{SCHEMA}].[{tabla}]",
        engine
    )
    df['EAN_Numerador']   = df['EAN_Numerador'].astype(str).str.strip()
    df['EAN_Denominador'] = df['EAN_Denominador'].astype(str).str.strip()
    df['Orden'] = pd.to_numeric(df['Orden'], errors='coerce').astype('Int64')
    return df


def leer_precio_deflactado(engine):
    """Lee la tabla de precio deflactado por EAN desde DB."""
    df = pd.read_sql(
        f"SELECT EAN, Precio_Deflactado_KG FROM [{SCHEMA}].[{TABLA_DEFLACTADO}]",
        engine
    )
    df['EAN'] = df['EAN'].astype(str).str.strip()
    df['Precio_Deflactado_KG'] = pd.to_numeric(df['Precio_Deflactado_KG'], errors='coerce')
    return df.drop_duplicates('EAN')


def leer_kg_u(engine_mds):
    """
    Carga el factor KG/U por EAN desde el maestro de MDS_PRODU. Necesario
    para convertir precio unitario a precio por kilogramo:
      PDG_KG = Precio_BASE / (KG / U)
    """
    df = pd.read_sql("""
        SELECT EAN, [KG / U] AS KG_U
        FROM [mdm].[ICO_EANs_mercado]
    """, engine_mds)
    df['EAN']  = df['EAN'].astype(str).str.strip()
    df['KG_U'] = pd.to_numeric(df['KG_U'], errors='coerce')
    return df[['EAN', 'KG_U']].drop_duplicates('EAN')


def agregar_price_index_nacional(df_pairs, df_deflactado):
    """
    Price_Index_Nacional = PrecioDeflactado[EAN_Num] / PrecioDeflactado[EAN_Den].
    Estatico: no varia por fecha, subcanal ni cliente.
    """
    defl = df_deflactado.set_index('EAN')['Precio_Deflactado_KG']
    df = df_pairs.copy()
    df['Price_Index_Nacional'] = df.apply(
        lambda r: (
            defl.get(r['EAN_Numerador'], np.nan) / defl.get(r['EAN_Denominador'], np.nan)
            if defl.get(r['EAN_Denominador'], 0) > 0 else np.nan
        ),
        axis=1
    )
    return df


def preparar_individuales_pi(df_gold, fuente, df_kg_u):
    """
    Toma el detalle por PDV ya leido de Gold (mismo df_gold que usan las
    metricas simples) y lo prepara para Price Index: precio por KG a nivel
    de boca individual, filtrado a una sola Fuente.
    """
    df = df_gold[df_gold['Fuente'] == fuente].copy()
    df = df.merge(df_kg_u, on='EAN', how='left')
    kgu = df['KG_U'].replace(0, np.nan)
    df['PDG_KG'] = df['Precio_BASE'] / kgu
    df = df.dropna(subset=['PDG_KG'])
    return df


def calcular_pi_por_pdv(df_pairs, df_ind):
    """
    Cociente del Price Index a nivel de boca individual, mismo PDV para
    numerador y denominador:
      PI_PDV = PDG_KG[EAN_Num, PDV] / PDG_KG[EAN_Den, PDV]

    El join usa solo Codigo_PDV + Fecha como clave (igual que
    platinum_price_index_copi/snp.ipynb) -- NO se incluye Descripcion_Cliente
    ni Subcanal en la clave de join a proposito: en COPI Descripcion_Cliente
    es siempre nulo, y usarlo como clave dependeria del comportamiento de
    pandas al matchear NaN==NaN en un merge, que es fragil. Provincia,
    Subcanal y Descripcion_Cliente los aporta el lado del numerador (misma
    boca), no hace falta traerlos de nuevo del denominador.

    Devuelve el detalle por boca (con PI_PDV) para reagregarlo por mediana
    al grano de 4 ejes que corresponda.
    """
    df_num = df_ind.rename(columns={'EAN': 'EAN_Numerador', 'PDG_KG': 'PDG_KG_Num'})
    df_den = df_ind.rename(columns={'EAN': 'EAN_Denominador', 'PDG_KG': 'PDG_KG_Den'})

    df = df_pairs.merge(df_num, on='EAN_Numerador', how='inner')
    df = df.merge(
        df_den[['EAN_Denominador', 'Codigo_PDV', 'Fecha', 'PDG_KG_Den']],
        on=['EAN_Denominador', 'Codigo_PDV', 'Fecha'],
        how='inner'
    )
    df['PI_PDV'] = np.where(df['PDG_KG_Den'] > 0, df['PDG_KG_Num'] / df['PDG_KG_Den'], np.nan)
    return df


def _agregar_pi(df_boca, claves):
    return (
        df_boca.groupby(claves, observed=True)
        .agg(
            Price_Index=('PI_PDV', 'median'),
            PDG_KG_Num=('PDG_KG_Num', 'median'),
            PDG_KG_Den=('PDG_KG_Den', 'median'),
            Cantidad_PDV=('Codigo_PDV', 'nunique'),
            Cantidad_Registros=('PI_PDV', 'size'),
        )
        .reset_index()
    )


def construir_grano_subcanal_pi(df_boca_copi):
    """Price Index al grano Subcanal (4 categorias) -- solo COPI, espejo de construir_grano_subcanal."""
    claves = ['Fecha', 'Provincia', 'Subcanal'] + PAIR_KEYS
    g = _agregar_pi(df_boca_copi, claves)
    g['Nivel_Cliente'] = 'Subcanal'
    g['Cliente']       = TODOS
    g['Fuente']        = 'COPI'
    return g


def construir_grano_cliente_pi(df_boca_snp):
    """Price Index al grano Cliente (cadena, Canal Moderno) -- solo SNP, espejo de construir_grano_cliente."""
    d = df_boca_snp[df_boca_snp['Subcanal'].isin(SUBCANALES_CANAL_MODERNO)]
    claves = ['Fecha', 'Provincia', 'Subcanal', 'Descripcion_Cliente'] + PAIR_KEYS
    g = _agregar_pi(d, claves)
    g = g.rename(columns={'Descripcion_Cliente': 'Cliente'})
    g['Nivel_Cliente'] = 'Cliente'
    g['Fuente']        = 'SNP'
    return g


def construir_bloque_price_index(df_gold, engine, engine_mds):
    """
    Arma el bloque Price Index completo (Subcanal desde COPI + Cliente desde
    SNP) al grano de 4 ejes -- misma formula validada de
    platinum_price_index_copi/snp.ipynb (mediana de cocientes por PDV),
    reagregada a esta granularidad en vez de la geografica/cadena original.

    Devuelve DataFrame vacio (con aviso) si faltan las tablas de referencia
    de pares -- gap de gobierno ya documentado en HORUS_ROADMAP.md, no debe
    tumbar el resto de la corrida.
    """
    faltantes = [t for t in (TABLA_PAIRS_COPI, TABLA_PAIRS_SNP, TABLA_DEFLACTADO)
                 if not _tabla_existe(engine, t)]
    if faltantes:
        print(f"[AVISO] No se encontraron las tablas de referencia {faltantes} -- "
              f"se omite el bloque Price Index.")
        return pd.DataFrame()

    print("[INFO] Leyendo pares, precio deflactado y KG/U para Price Index...")
    df_deflactado = leer_precio_deflactado(engine)
    df_kg_u       = leer_kg_u(engine_mds)

    df_pairs_copi = agregar_price_index_nacional(leer_pairs(engine, TABLA_PAIRS_COPI), df_deflactado)
    df_pairs_snp  = agregar_price_index_nacional(leer_pairs(engine, TABLA_PAIRS_SNP),  df_deflactado)

    df_ind_copi = preparar_individuales_pi(df_gold, 'COPI', df_kg_u)
    df_ind_snp  = preparar_individuales_pi(df_gold, 'SNP',  df_kg_u)

    df_boca_copi = calcular_pi_por_pdv(df_pairs_copi, df_ind_copi)
    df_boca_snp  = calcular_pi_por_pdv(df_pairs_snp,  df_ind_snp)

    g_subcanal = construir_grano_subcanal_pi(df_boca_copi) if not df_boca_copi.empty else pd.DataFrame()
    g_cliente  = construir_grano_cliente_pi(df_boca_snp)   if not df_boca_snp.empty  else pd.DataFrame()

    out = pd.concat([g_subcanal, g_cliente], ignore_index=True)
    if out.empty:
        print("[AVISO] Price Index no genero filas (sin cruces PDV entre pares y precios).")
        return out

    out['Tipo_Metrica'] = 'Price_Index'
    out = agregar_region_v2(out)
    out = out.replace([float('inf'), float('-inf')], np.nan)
    return out.reindex(columns=COLUMNAS_FINAL)


In [4]:
# =============================================================================
# FUNCIONES — METRICAS SIMPLES (Precio_Promedio / Precio_Mediana)
# =============================================================================

def leer_gold_comparativo():
    """
    Lee de Gold_Comparativo lo necesario para los 4 ejes, incluyendo EAN
    (requerido tambien por el bloque Price Index mas abajo, que reusa este
    mismo DataFrame en vez de leer Gold dos veces). Filtra registros sin
    Precio_BASE valido o sin las claves minimas de agrupacion (SKU,
    Provincia, Subcanal) -- sin esas tres, la fila no puede ubicarse en
    ninguno de los dos grafos de agregacion de mas abajo.
    """
    print("[INFO] Leyendo PRC.Gold_Comparativo...")
    query = f"""
        SELECT Fuente, Fecha, EAN, Cod_SKU, Descripcion_SKU, Provincia, Subcanal,
               Cod_Cliente, Descripcion_Cliente, Codigo_PDV, Precio_BASE
        FROM [{SCHEMA}].[{TABLA_GOLD}]
        WHERE Precio_BASE IS NOT NULL AND Precio_BASE > 0
          AND Cod_SKU  IS NOT NULL
          AND Provincia IS NOT NULL
          AND Subcanal  IS NOT NULL
    """
    df = pd.read_sql(query, engine)
    print(f"   {len(df):,} filas leidas.")

    df['Fecha']    = pd.to_datetime(df['Fecha'], errors='coerce')
    df['EAN']      = df['EAN'].astype(str).str.strip()
    df['Cod_SKU']  = df['Cod_SKU'].astype(str).str.strip()
    df['Provincia'] = df['Provincia'].astype(str).str.strip()
    # SNP trae Subcanal con prefijo numerico de orden ("1. Supermercados",
    # "2. Mayoristas" -- confirmado 2026-08-04 contra Gold_Comparativo real).
    # COPI no lo trae. Se normaliza aca, en el unico punto de entrada de
    # datos, para que el resto del pipeline (filtro de Canal Moderno y la
    # columna Subcanal de salida) trabaje sobre un valor comparable entre
    # las dos fuentes.
    df['Subcanal'] = (
        df['Subcanal'].astype(str).str.strip()
        .str.replace(r'^\d+\.\s*', '', regex=True)
    )
    return df


def _agregar(df, claves):
    return (
        df.groupby(claves, observed=True)
          .agg(
              Precio_Promedio=('Precio_BASE', 'mean'),
              Precio_Mediana=('Precio_BASE', 'median'),
              Cantidad_PDV=('Codigo_PDV', 'nunique'),
              Cantidad_Registros=('Precio_BASE', 'size'),
          )
          .reset_index()
    )


def construir_grano_subcanal(df):
    """
    Eje Cliente = Subcanal, para las 4 categorias. Solo COPI releva AASS+Petit,
    asi que esta apertura sale exclusivamente de COPI (aunque SNP tambien
    tenga Subcanal, usar solo COPI evita mezclar dos paneles distintos en el
    mismo nivel de grano).
    """
    d = df[df['Fuente'] == 'COPI'].copy()
    g = _agregar(d, ['Fecha', 'Cod_SKU', 'Descripcion_SKU', 'Provincia', 'Subcanal'])
    g['Nivel_Cliente'] = 'Subcanal'
    g['Cliente']       = TODOS
    g['Fuente']        = 'COPI'
    return g


def construir_grano_cliente(df):
    """
    Eje Cliente = Cliente (cadena), solo para los Subcanales de Canal Moderno.
    Unicamente SNP identifica cadena (Descripcion_Cliente); se descartan filas
    sin cliente identificado (bocas SNP que aun no estan en el maestro).

    Nota (confirmado 2026-08-04 contra datos reales): SNP hoy solo releva
    "Supermercados" y "Mayoristas" a nivel cadena -- no hay filas de
    "E-Commerce" en el panel SNP todavia. El filtro deja el tercer valor
    igual (por si se suma a futuro), pero el dato de hoy no lo va a traer.
    """
    d = df[
        (df['Fuente'] == 'SNP')
        & df['Subcanal'].isin(SUBCANALES_CANAL_MODERNO)
        & df['Descripcion_Cliente'].notna()
    ].copy()
    g = _agregar(
        d, ['Fecha', 'Cod_SKU', 'Descripcion_SKU', 'Provincia', 'Subcanal', 'Descripcion_Cliente']
    )
    g = g.rename(columns={'Descripcion_Cliente': 'Cliente'})
    g['Nivel_Cliente'] = 'Cliente'
    g['Fuente']        = 'SNP'
    return g


def construir_bloque_precio_simple(df):
    """
    Concatena las dos aperturas (Subcanal desde COPI, Cliente desde SNP) de
    la metrica simple (Precio_Promedio/Precio_Mediana) y agrega el eje
    geografico placeholder.
    """
    g_subcanal = construir_grano_subcanal(df)
    g_cliente  = construir_grano_cliente(df)

    out = pd.concat([g_subcanal, g_cliente], ignore_index=True)
    out['Tipo_Metrica'] = 'Precio_Simple'
    out = agregar_region_v2(out)
    return out.reindex(columns=COLUMNAS_FINAL)


def imprimir_resumen(df):
    print("\n[INFO] RESUMEN PLATINUM 4 EJES:")
    print(f"   Total filas:               {len(df):,}")
    for (metrica, nivel), sub in df.groupby(['Tipo_Metrica', 'Nivel_Cliente'], observed=True):
        print(f"   Tipo_Metrica={metrica:<13} Nivel_Cliente={nivel:<10} filas={len(sub):,}  "
              f"semanas={sub['Fecha'].nunique():,}")


In [5]:
# =============================================================================
# PIPELINE PRINCIPAL
# =============================================================================

def run_platinum_4ejes():
    print(f"--- INICIO PLATINUM 4 EJES: {datetime.datetime.now()} ---\n")

    df_gold = leer_gold_comparativo()

    df_simple = construir_bloque_precio_simple(df_gold)
    df_pi     = construir_bloque_price_index(df_gold, engine, engine_mds)

    df_out = pd.concat([df_simple, df_pi], ignore_index=True)
    imprimir_resumen(df_out)

    CHUNK_SIZE = 5000
    total = len(df_out)
    print(f"\n[INFO] Cargando {total:,} filas a {SCHEMA}.{TABLA_OUT}...")

    if total == 0:
        df_out.to_sql(TABLA_OUT, con=engine, schema=SCHEMA, if_exists='replace', index=False)
    else:
        cargadas = 0
        for i, start in enumerate(range(0, total, CHUNK_SIZE)):
            chunk = df_out.iloc[start:start + CHUNK_SIZE]
            chunk.to_sql(
                TABLA_OUT, con=engine, schema=SCHEMA,
                if_exists='replace' if i == 0 else 'append',
                index=False
            )
            cargadas += len(chunk)
            print(f"   -> {cargadas:,} / {total:,} filas ({cargadas / total * 100:.1f}%)")

    print(f"[OK] {SCHEMA}.{TABLA_OUT} cargada correctamente.")
    print(f"\n--- PLATINUM 4 EJES FINALIZADO: {datetime.datetime.now()} ---")


if __name__ == "__main__":
    run_platinum_4ejes()


--- INICIO PLATINUM 4 EJES: 2026-09-03 08:23:25.707623 ---

[INFO] Leyendo PRC.Gold_Comparativo...
   437,581 filas leidas.
[INFO] Leyendo pares, precio deflactado y KG/U para Price Index...

[INFO] RESUMEN PLATINUM 4 EJES:
   Total filas:               79,886
   Tipo_Metrica=Precio_Simple Nivel_Cliente=Cliente    filas=5,736  semanas=99
   Tipo_Metrica=Precio_Simple Nivel_Cliente=Subcanal   filas=66,530  semanas=245
   Tipo_Metrica=Price_Index   Nivel_Cliente=Subcanal   filas=7,620  semanas=242

[INFO] Cargando 79,886 filas a PRC.platinum_4ejes...
   -> 5,000 / 79,886 filas (6.3%)
   -> 10,000 / 79,886 filas (12.5%)
   -> 15,000 / 79,886 filas (18.8%)
   -> 20,000 / 79,886 filas (25.0%)
   -> 25,000 / 79,886 filas (31.3%)
   -> 30,000 / 79,886 filas (37.6%)
   -> 35,000 / 79,886 filas (43.8%)
   -> 40,000 / 79,886 filas (50.1%)
   -> 45,000 / 79,886 filas (56.3%)
   -> 50,000 / 79,886 filas (62.6%)
   -> 55,000 / 79,886 filas (68.8%)
   -> 60,000 / 79,886 filas (75.1%)
   -> 65,000 / 